In [2]:
import pandas as pd
from pyliftover import LiftOver

manifest_path = "/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins/test_code/valid/valid_mqtl/infinium-methylationepic-v-1-0-b5-manifest-file.csv"  # 已解压的 CSV 路径
# 跳过前 7 行注释，根据实际文件可微调
epict_manifest = pd.read_csv(
    manifest_path,
    skiprows=7,
    usecols=["IlmnID", "CHR", "MAPINFO"]
)
# 重命名以标准化列名
epict_manifest.columns = ["cpg_id", "chr_hg19", "pos_hg19"]
epict_manifest.set_index("cpg_id", inplace=True)

In [3]:
lo = LiftOver("hg19", "hg38")  # 默认从 hg19 -> hg38

In [4]:
# df = pd.read_csv("/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/original/GTEX/BreastMammaryTissue.regular.perm.fdr.txt",
#                         sep='\t')
df = pd.read_csv("/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/original/EPIGEN/EPIC.txt",
                        sep='\t')
df.rename(columns={"CpG": "cpg_id"}, inplace=True)
df = df.join(epict_manifest, on="cpg_id", how="left")  # 左连接，缺失值为 NaN

In [5]:
def liftover_pos(row):
    if pd.isna(row["chr_hg19"]) or pd.isna(row["pos_hg19"]):
        return pd.Series([None, None])
    chrom = f"chr{row['chr_hg19']}"
    # MAPINFO 是 1-based, pyliftover 也期望 1-based
    result = lo.convert_coordinate(chrom, int(row["pos_hg19"]))
    if not result:
        return pd.Series([None, None])
    # 取第一个匹配，result 返回 [(chrom, pos, strand, score), ...]
    new_chrom, new_pos, strand, score = result[0]
    return pd.Series([new_chrom.replace("chr", ""), new_pos])
# 执行批量 liftover
df[['chr_hg38', 'pos_hg38']] = df.apply(liftover_pos, axis=1)

# same to CpG pos
def liftover_CpG_pos(row):
    if pd.isna(row["CpG chr"]) or pd.isna(row["CpG pos"]):
        return pd.Series([None, None])
    chrom = f"chr{row['CpG chr']}"
    result = lo.convert_coordinate(chrom, int(row["CpG pos"]))
    if not result:
        return pd.Series([None, None])
    new_chrom, new_pos, strand, score = result[0]
    return pd.Series([new_chrom.replace("chr", ""), new_pos])

# Apply liftover to SNP positions
df[['CpG_chr_hg38', 'CpG_pos_hg38']] = df.apply(liftover_CpG_pos, axis=1)

def liftover_SNP_pos(row):
    if pd.isna(row["SNP chr"]) or pd.isna(row["SNP pos"]):
        return pd.Series([None, None])
    chrom = f"chr{row['SNP chr']}"
    result = lo.convert_coordinate(chrom, int(row["SNP pos"]))
    if not result:
        return pd.Series([None, None])
    new_chrom, new_pos, strand, score = result[0]
    return pd.Series([new_chrom.replace("chr", ""), new_pos])

# Apply liftover to SNP positions
df[['SNP_chr_hg38', 'SNP_pos_hg38']] = df.apply(liftover_SNP_pos, axis=1)

In [6]:
df.head()

,cpg_id,SNP,CpG chr,CpG pos,SNP chr,SNP pos,Other Allele,Effect Allele,MAF,Beta,...,FDR,Cis/Trans,chr_hg19,pos_hg19,chr_hg38,pos_hg38,CpG_chr_hg38,CpG_pos_hg38,SNP_chr_hg38,SNP_pos_hg38
0,cg06325811,1:753405_C_A,1,796328.0,1,753405,C,A,0.158,0.371869,...,0.0,cis,1,796328.0,1,860948.0,1,860948.0,1,818025.0
1,cg16619049,1:798959_G_A,1,805541.0,1,798959,G,A,0.214,0.769962,...,0.0,cis,1,805541.0,1,870161.0,1,870161.0,1,863579.0
2,cg13938959,1:838329_G_GC,1,834183.0,1,838329,G,GC,0.199,0.673932,...,0.0,cis,1,834183.0,1,898803.0,1,898803.0,1,902949.0
3,cg12445832,1:834999_G_A,1,834295.0,1,834999,G,A,0.203,-0.386778,...,0.0,cis,1,834295.0,1,898915.0,1,898915.0,1,899619.0
4,cg04195702,1:842057_A_AAACTCAGCTGCCTCTCCCCTTC,1,838486.0,1,842057,A,AAACTCAGCTGCCTCTCCCCTTC,0.279,-0.317762,...,0.0,cis,1,838486.0,1,903106.0,1,903106.0,1,906677.0


In [7]:
maf_thresh     = 0.05  # 最低次等位基因频率
beta_thresh    = 1  # 最小绝对效应值
se_thresh      = 0.05  # 最大效应标准误
p_nominal_max  = 1e-5  # 最大名义p值
fdr_thresh     = 0.05  # 最大 FDR

filtered = df[
    (df["MAF"] >= maf_thresh) &
    (df["Beta"].abs() >= beta_thresh) &
    (df["SE"] <= se_thresh) &
    (df["P"] <= p_nominal_max) &
    (df["FDR"] <= fdr_thresh) &
    (df["Effect Allele"].str.len() == 1) &
    (df["Other Allele"].str.len() == 1) & 
    ((df['CpG_pos_hg38'] - df['SNP_pos_hg38']).abs() < 9000)
]

print(f"筛选后保留 {len(filtered)} 条 mQTL 记录")

筛选后保留 1002 条 mQTL 记录


In [8]:
filtered = filtered.copy()

In [9]:
filtered['SNP_region_start'] = filtered['SNP_pos_hg38']
filtered['SNP_region_end'] = filtered['SNP_pos_hg38']
filtered['SNP_ref'] = filtered['SNP'].str.split('_').str[1]
filtered['SNP_alt'] = filtered['SNP'].str.split('_').str[2]
filtered['chrom'] = filtered['CpG chr']
filtered['CPG_region_start'] = filtered['pos_hg38']
filtered['CPG_region_end'] = filtered['pos_hg38']
filtered['effect_size'] = filtered['Beta']

In [13]:
filtered[171:178]

,cpg_id,SNP,CpG chr,CpG pos,SNP chr,SNP pos,Other Allele,Effect Allele,MAF,Beta,...,SNP_chr_hg38,SNP_pos_hg38,SNP_region_start,SNP_region_end,SNP_ref,SNP_alt,chrom,CPG_region_start,CPG_region_end,effect_size
39056,cg07471365,2:242737341_T_C,2,242737372.0,2,242737341,T,C,0.3180,1.021764,...,2,241797926.0,241797926.0,241797926.0,T,C,2,241797957.0,241797957.0,1.021764
39063,cg10833299,2:242754192_A_G,2,242749776.0,2,242754192,A,G,0.3470,1.243215,...,2,241813054.0,241813054.0,241813054.0,A,G,2,241808681.0,241808681.0,1.243215
39077,cg26207766,2:242763911_A_G,2,242763794.0,2,242763911,A,G,0.3500,1.213532,...,2,241821726.0,241821726.0,241821726.0,A,G,2,241821609.0,241821609.0,1.213532
39078,cg03455424,2:242755465_C_G,2,242763982.0,2,242755465,C,G,0.3400,1.309802,...,2,241814268.0,241814268.0,241814268.0,C,G,2,241821797.0,241821797.0,1.309802
39289,cg21594961,3:3152930_G_T,3,3152916.0,3,3152930,G,T,0.2640,1.332524,...,3,3111246.0,3111246.0,3111246.0,G,T,3,3111232.0,3111232.0,1.332524
39808,cg04768501,3:9964582_G_A,3,9956823.0,3,9964582,G,A,0.4820,1.005452,...,3,9922898.0,9922898.0,9922898.0,G,A,3,9915139.0,9915139.0,1.005452
40832,cg15204713,3:15536634_G_A,3,15540160.0,3,15536634,G,A,0.0719,1.239085,...,3,15495127.0,15495127.0,15495127.0,G,A,3,15498653.0,15498653.0,1.239085


In [11]:
filtered_save = filtered.copy()
filtered_save = filtered_save[['chrom', 'SNP_region_start', 'SNP_region_end', 'SNP_ref', 'SNP_alt', 'CPG_region_start', 'CPG_region_end', 'effect_size']]
# Convert chrom to string and add 'chr' prefix
filtered_save['chrom'] = filtered_save['chrom'].astype(str)
filtered_save['chrom'] = "chr" + filtered_save['chrom']
# filtered_save = filtered_save[(filtered_save['SNP_region_start'] - filtered_save['CPG_region_start']).abs() < 9000]
import os
if not os.path.exists(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/EPIGEN/"):
    os.makedirs(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/EPIGEN/", exist_ok=True)
filtered_save.to_csv(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/EPIGEN/EPIC.csv", index=False)

In [14]:
# df = pd.read_csv("/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/original/GTEX/BreastMammaryTissue.regular.perm.fdr.txt",
#                         sep='\t')
df = pd.read_csv("/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/original/EPIGEN/skin_conditional_meQTL.txt",
                        sep='\t')
df.rename(columns={"CpG": "cpg_id"}, inplace=True)
df = df.join(epict_manifest, on="cpg_id", how="left")  # 左连接，缺失值为 NaN

def liftover_pos(row):
    if pd.isna(row["chr_hg19"]) or pd.isna(row["pos_hg19"]):
        return pd.Series([None, None])
    chrom = f"chr{row['chr_hg19']}"
    # MAPINFO 是 1-based, pyliftover 也期望 1-based
    result = lo.convert_coordinate(chrom, int(row["pos_hg19"]))
    if not result:
        return pd.Series([None, None])
    # 取第一个匹配，result 返回 [(chrom, pos, strand, score), ...]
    new_chrom, new_pos, strand, score = result[0]
    return pd.Series([new_chrom.replace("chr", ""), new_pos])
# 执行批量 liftover
df[['chr_hg38', 'pos_hg38']] = df.apply(liftover_pos, axis=1)

# same to CpG pos
def liftover_CpG_pos(row):
    if pd.isna(row["CpG chr"]) or pd.isna(row["CpG pos"]):
        return pd.Series([None, None])
    chrom = f"chr{row['CpG chr']}"
    result = lo.convert_coordinate(chrom, int(row["CpG pos"]))
    if not result:
        return pd.Series([None, None])
    new_chrom, new_pos, strand, score = result[0]
    return pd.Series([new_chrom.replace("chr", ""), new_pos])

# Apply liftover to CpG positions
df[['CpG_chr_hg38', 'CpG_pos_hg38']] = df.apply(liftover_CpG_pos, axis=1)

def liftover_SNP_pos(row):
    if pd.isna(row["SNP chr"]) or pd.isna(row["SNP pos"]):
        return pd.Series([None, None])
    chrom = f"chr{row['SNP chr']}"
    result = lo.convert_coordinate(chrom, int(row["SNP pos"]))
    if not result:
        return pd.Series([None, None])
    new_chrom, new_pos, strand, score = result[0]
    return pd.Series([new_chrom.replace("chr", ""), new_pos])

# Apply liftover to SNP positions
df[['SNP_chr_hg38', 'SNP_pos_hg38']] = df.apply(liftover_SNP_pos, axis=1)

In [56]:
df.head()

,cpg_id,SNP,CpG chr,CpG pos,SNP chr,SNP pos,Effect Allele,Other Allele,MAF,Beta,...,FDR,Cis/Trans,chr_hg19,pos_hg19,chr_hg38,pos_hg38,CpG_chr_hg38,CpG_pos_hg38,SNP_chr_hg38,SNP_pos_hg38
0,cg16352085,chr4:168502590:SNP,1,859719,4,168502590,C,T,0.3629,0.626246,...,1.752572e-10,trans,1,859719.0,1,924339.0,1,924339.0,4,167581439
1,cg22114309,chr2:130993624:SNP,1,895869,2,130993624,G,A,0.4937,-0.698658,...,1.184738e-19,trans,1,895869.0,1,960489.0,1,960489.0,2,130236051
2,cg23207077,chr1:1019175:SNP,1,905988,1,1019175,C,G,0.3008,0.395880,...,4.143302e-04,cis,1,905988.0,1,970608.0,1,970608.0,1,1083795
3,cg25107893,chr1:1017341:SNP,1,911601,1,1017341,G,T,0.2982,0.718644,...,8.100419e-17,cis,1,911601.0,1,976221.0,1,976221.0,1,1081961
4,cg05505459,chr1:1017341:SNP,1,911995,1,1017341,G,T,0.2982,0.718599,...,8.147754e-17,cis,1,911995.0,1,976615.0,1,976615.0,1,1081961


In [15]:
maf_thresh     = 0.05  # 最低次等位基因频率
beta_thresh    = 0.5  # 最小绝对效应值
se_thresh      = 0.05  # 最大效应标准误
p_nominal_max  = 1e-5  # 最大名义p值
fdr_thresh     = 0.05  # 最大 FDR

filtered = df[
    (df["MAF"] >= maf_thresh) &
    (df["Beta"].abs() >= beta_thresh) &
    (df["SE"] <= se_thresh) &
    (df["P"] <= p_nominal_max) &
    (df["FDR"] <= fdr_thresh) &
    (df["Effect Allele"].str.len() == 1) &
    (df["Other Allele"].str.len() == 1) & 
    ((df['CpG_pos_hg38'] - df['SNP_pos_hg38']).abs() < 9000)
].copy()

print(f"筛选后保留 {len(filtered)} 条 mQTL 记录")

筛选后保留 455 条 mQTL 记录


In [18]:
filtered['SNP_region_start'] = filtered['SNP_pos_hg38']
filtered['SNP_region_end'] = filtered['SNP_pos_hg38']
# filtered['SNP_ref'] = filtered['Effect Allele']
# filtered['SNP_alt'] = filtered['Other Allele']
filtered['SNP_ref'] = filtered['Other Allele']
filtered['SNP_alt'] = filtered['Effect Allele']
filtered['chrom'] = filtered['CpG chr']
filtered['CPG_region_start'] = filtered['pos_hg38']
filtered['CPG_region_end'] = filtered['pos_hg38']
filtered['effect_size'] = filtered['Beta']

In [93]:
filtered

,cpg_id,SNP,CpG chr,CpG pos,SNP chr,SNP pos,Effect Allele,Other Allele,MAF,Beta,...,SNP_chr_hg38,SNP_pos_hg38,SNP_region_start,SNP_region_end,SNP_ref,SNP_alt,chrom,CPG_region_start,CPG_region_end,effect_size
440,cg14790876,chr1:2798335:SNP,1,2798280,1,2798335,T,G,0.4975,1.07609,...,1,2881770,2881770,2881770,T,G,1,2881715.0,2881715.0,1.07609
914,cg13928473,chr1:6063655:SNP,1,6063654,1,6063655,G,A,0.4949,1.13977,...,1,6003595,6003595,6003595,G,A,1,6003594.0,6003594.0,1.13977
1009,cg18116088,chr1:6657240:SNP,1,6657059,1,6657240,T,C,0.3439,-1.18710,...,1,6597180,6597180,6597180,T,C,1,NaN,NaN,-1.18710
1040,cg20434152,chr1:7120915:SNP,1,7120926,1,7120915,A,G,0.3947,-1.10131,...,1,7060855,7060855,7060855,A,G,1,7060866.0,7060866.0,-1.10131
1246,cg17425144,chr1:10567711:SNP,1,10567563,1,10567711,C,T,0.4746,-1.05574,...,1,10507654,10507654,10507654,C,T,1,10507506.0,10507506.0,-1.05574
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82673,cg11247378,chr22:39784845:SNP,22,39784982,22,39784845,G,C,0.3173,1.14975,...,22,39388840,39388840,39388840,G,C,22,39388977.0,39388977.0,1.14975
83016,cg01808030,chr22:45809624:SNP,22,45809952,22,45809624,A,C,0.3871,0.99786,...,22,45413743,45413743,45413743,A,C,22,45414071.0,45414071.0,0.99786
83123,cg17099656,chr22:47135205:SNP,22,47135171,22,47135205,C,A,0.4619,1.01302,...,22,46739308,46739308,46739308,C,A,22,46739274.0,46739274.0,1.01302
83124,cg16154810,chr22:47135272:SNP,22,47135258,22,47135272,A,G,0.4594,1.00411,...,22,46739375,46739375,46739375,A,G,22,46739361.0,46739361.0,1.00411


In [19]:
filtered_save = filtered.copy()
filtered_save = filtered_save[['chrom', 'SNP_region_start', 'SNP_region_end', 'SNP_ref', 'SNP_alt', 'CPG_region_start', 'CPG_region_end', 'effect_size']]
filtered_save = filtered_save[filtered_save['CPG_region_start'].notna()]
filtered_save = filtered_save[(filtered_save['SNP_region_start'] - filtered_save['CPG_region_start']).abs() < 9000]
filtered_save['chrom'] = filtered_save['chrom'].astype(str)
filtered_save['chrom'] = "chr" + filtered_save['chrom']
import os
if not os.path.exists(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/EPIGEN/"):
    os.makedirs(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/EPIGEN/", exist_ok=True)
filtered_save.to_csv(f"/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins_gate/meqtl/dataset/processed/EPIGEN/skin.csv", index=False)

In [94]:
filtered_save

,chrom,SNP_region_start,SNP_region_end,SNP_ref,SNP_alt,CPG_region_start,CPG_region_end,effect_size
440,chr1,2881770,2881770,T,G,2881715.0,2881715.0,1.07609
914,chr1,6003595,6003595,G,A,6003594.0,6003594.0,1.13977
1040,chr1,7060855,7060855,A,G,7060866.0,7060866.0,-1.10131
1246,chr1,10507654,10507654,C,T,10507506.0,10507506.0,-1.05574
1618,chr1,17097638,17097638,T,C,17098029.0,17098029.0,1.11664
...,...,...,...,...,...,...,...,...
82673,chr22,39388840,39388840,G,C,39388977.0,39388977.0,1.14975
83016,chr22,45413743,45413743,A,C,45414071.0,45414071.0,0.99786
83123,chr22,46739308,46739308,C,A,46739274.0,46739274.0,1.01302
83124,chr22,46739375,46739375,A,G,46739361.0,46739361.0,1.00411


In [90]:
import sys

sys.path.append("/archive/bioinformatics/Zhou_lab/shared/jjin/project/steins/test_code")
from selene_mini import *
genome = Genome(input_path='/archive/bioinformatics/Zhou_lab/shared/jzhou/graphseq/Homo_sapiens.GRCh38.dna.primary_assembly.fa')

In [91]:
genome.get('chr1', 2881770-2, 2881770+2)

array([[0., 1., 0., 0.],
       [0., 0., 0., 1.],
       [0., 1., 0., 0.],
       [0., 0., 0., 1.]])

In [92]:
genome.get('chr1', 6003595-2, 6003595+2)

array([[0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 0., 1.],
       [0., 0., 0., 1.]])

In [95]:
genome.get('chr1', 7060855-2, 7060855+2)

array([[0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.],
       [1., 0., 0., 0.]])

In [96]:
genome.get('chr1', 10507654-2, 10507654+2)

array([[0., 0., 1., 0.],
       [0., 1., 0., 0.],
       [0., 0., 1., 0.],
       [0., 0., 1., 0.]])